# 📄 RAG Pipeline with LangChain, FAISS & HuggingFace

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline from scratch. It is organized into three stages:

| Stage | File | Purpose |
|-------|------|---------|
| 1️⃣ Ingestion | `ingest.py` | Load PDF → Split → Embed → Store in FAISS |
| 2️⃣ Query | `query.py` | Load FAISS index → Semantic similarity search |
| 3️⃣ RAG Generation | `rag.py` | Retrieve context → Feed to LLM → Generate answer |

**Tech Stack:** `LangChain` · `FAISS` · `HuggingFace Transformers` · `sentence-transformers` · `Flan-T5`

---


## 🔧 Step 0 — Install Dependencies

Install all required libraries before running the pipeline.


In [1]:
!pip install langchain langchain-community faiss-cpu sentence-transformers transformers pypdf -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## 📥 Stage 1 — Document Ingestion (`ingest.py`)

### What happens here?
1. **Load** the PDF using `PyPDFLoader` — converts each page into a LangChain `Document` object.
2. **Split** the text into smaller overlapping chunks using `RecursiveCharacterTextSplitter`.
   - `chunk_size=500` — each chunk holds ~500 characters
   - `chunk_overlap=50` — 50-character overlap ensures context isn't lost at boundaries
3. **Embed** each chunk using the `all-MiniLM-L6-v2` sentence-transformer model — converts text → 384-dimensional vectors.
4. **Store** all vectors in a **FAISS** index and save it locally for reuse.

> 💡 FAISS (Facebook AI Similarity Search) enables blazing-fast approximate nearest-neighbour search over millions of vectors.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# ── Load PDF ──────────────────────────────────────────────────
# Replace with your own PDF file path
loader = PyPDFLoader("attention_all_u_need.pdf")
docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"\nSample content (first 300 chars of page 1):\n{docs[0].page_content[:300]}")



Total pages loaded: 15

Sample content (first 300 chars of page 1):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [3]:
# ── Chunk the documents ───────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")


Total chunks created: 93

Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── Embed & index ─────────────────────────────────────────────
# Downloads ~90MB model on first run
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(chunks, embedding)

# ── Save FAISS index to disk ──────────────────────────────────
db.save_local("faiss_index")

print("✅ FAISS index saved to ./faiss_index/")


C:\Users\KAUSHIK\AppData\Local\Temp\ipykernel_10624\1461050294.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


✅ FAISS index saved to ./faiss_index/


---

## 🔍 Stage 2 — Semantic Search / Query (`query.py`)

### What happens here?
1. **Load** the saved FAISS index back into memory.
2. Accept a **natural language query** from the user.
3. Perform **similarity search** — embeds the query, finds the top-k closest chunks in vector space.
4. Return and display the most relevant document chunks.

> 💡 `k=3` means we retrieve the 3 most semantically similar chunks to the query.


In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Reload the FAISS index ────────────────────────────────────
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True   # required for local FAISS indexes
)

print("✅ FAISS index loaded successfully.")


✅ FAISS index loaded successfully.


In [6]:
# ── Run a similarity search ───────────────────────────────────
# Change this query to test different questions
query = "What is the attention mechanism?"

docs = db.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n{'='*50}")
    print(f"Result {i+1}:")
    print(f"{'='*50}")
    print(doc.page_content)



Result 1:
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads. Best viewed in color.
13

Result 2:
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perform different tasks.
15

Result 3:
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Mul

---

## 🤖 Stage 3 — Full RAG Pipeline (`rag.py`)

### What happens here?
This is the complete **Retrieval-Augmented Generation** loop:

```
User Query
    │
    ▼
[FAISS Retriever] ── semantic search ──► Top-k Chunks (Context)
    │
    ▼
[Prompt Builder] ── wraps query + context into structured prompt
    │
    ▼
[Flan-T5 LLM] ── generates grounded answer from context only
    │
    ▼
Answer printed to user
```

**Key design decisions:**
- **Score filtering** (`score < 1.5`) — only includes chunks that are genuinely relevant; falls back to top-2 if nothing passes.
- **Context truncation** (`[:1000]`) — prevents token overflow in the LLM.
- **Structured prompt** — instructs the LLM to answer *only* from context, avoiding hallucination.
- **Flan-T5-base** — a free, open-source text2text model by Google; no API key needed.


In [7]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# ── Load embeddings + FAISS index ─────────────────────────────
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

print("✅ Embeddings and FAISS index ready.")


✅ Embeddings and FAISS index ready.


In [8]:
# ── Load LLM (Flan-T5-base — Free & Local) ────────────────────
# First run will download ~990MB model weights
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM loaded: google/flan-t5-base")


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ LLM loaded: google/flan-t5-base


C:\Users\KAUSHIK\AppData\Local\Temp\ipykernel_10624\1000539139.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [9]:
# ── Retrieval Function with Score Filtering ───────────────────
def retrieve_docs(query):
    """
    Retrieve the most relevant chunks for a query.
    - Uses FAISS similarity_search_with_score (L2 distance)
    - Filters out chunks with score >= 1.5 (too dissimilar)
    - Falls back to top-2 chunks if all are filtered out
    """
    docs_with_scores = db.similarity_search_with_score(query, k=3)

    # Keep only high-relevance chunks (lower L2 score = more similar)
    filtered_docs = [doc for doc, score in docs_with_scores if score < 1.5]

    # Fallback: if nothing passes the threshold, use top-2 anyway
    if not filtered_docs:
        filtered_docs = [doc for doc, _ in docs_with_scores[:2]]

    return filtered_docs[:2]

print("✅ retrieve_docs() function defined.")


✅ retrieve_docs() function defined.


In [10]:
# ── Prompt Builder ─────────────────────────────────────────────
def build_prompt(query, context):
    """
    Wraps the user query and retrieved context into a structured prompt
    that instructs the LLM to answer ONLY from the provided context.
    """
    return f"""
You are an AI tutor.

Instructions:
- Answer ONLY using the context below
- If the answer is not in the context, say: Not in document
- Be clear and structured
- Do not add extra knowledge

Context:
{context}

Question:
{query}

Answer in this format:
- Explanation:
- Key Points:
"""


In [11]:
# ── Single Query (Notebook-friendly, no while loop) ───────────
# Change this to test different questions
query = "What is multi-head attention?"

# Step 1: Retrieve relevant chunks
docs = retrieve_docs(query)

# Step 2: Show retrieved context
print("--- Retrieved Context ---")
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:\n{doc.page_content[:200]}")

# Step 3: Build context string
context = "\n\n".join([doc.page_content for doc in docs])
context = context[:1000]   # truncate to prevent token overflow

# Step 4: Build prompt
prompt = build_prompt(query, context)

# Step 5: Generate answer
response = llm.invoke(prompt)

print("\n" + "="*50)
print("Answer:")
print("="*50)
print(response)


--- Retrieved Context ---

Chunk 1:
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
descr

Chunk 2:
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
senten

Answer:
Attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence


---

## 🔁 Optional — Interactive Q&A Loop

Run the cell below to enter an interactive question-answering session. Type `exit` to quit.

> ⚠️ **Note:** `input()` works in Jupyter but may not work in all notebook environments (e.g., JupyterLite). Use the single-query cell above for guaranteed compatibility.


In [ ]:
# ── Interactive Q&A loop ──────────────────────────────────────
while True:
    query = input("\nAsk (or type 'exit' to quit): ").strip()

    if query.lower() == "exit":
        print("Goodbye!")
        break

    docs = retrieve_docs(query)

    print("\n--- Retrieved Context ---")
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc.page_content[:200]}")

    context = "\n\n".join([doc.page_content for doc in docs])
    context = context[:1000]

    prompt = build_prompt(query, context)
    response = llm.invoke(prompt)

    print("\n" + "="*50)
    print("Answer:")
    print("="*50)
    print(response)



--- Retrieved Context ---

Chunk 1:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

Chunk 2:
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output position

Answer:
model architecture

--- Retrieved Context ---

Chunk 1:
queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the
query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the
valu

Chunk 2:
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈ Rd, such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we


---

## 📌 Summary & Key Concepts

| Concept | Details |
|---------|---------|
| **RAG** | Combines retrieval (FAISS) + generation (LLM) to ground answers in real documents |
| **Chunking** | Splits large docs into overlapping pieces to preserve context at boundaries |
| **Embeddings** | `all-MiniLM-L6-v2` maps text → 384-dim vectors capturing semantic meaning |
| **FAISS** | Efficient vector store for approximate nearest-neighbour search |
| **Score filtering** | L2 distance < 1.5 ensures only genuinely relevant chunks are used |
| **Flan-T5** | Free, open-source seq2seq LLM — no API key required |
| **Prompt engineering** | Structured prompt restricts LLM to context, reducing hallucination |

### 🚀 Possible Extensions
- Swap Flan-T5 for a larger model (e.g., `flan-t5-large`, Mistral via Ollama)
- Add a Streamlit / Gradio UI for a chatbot interface
- Support multiple PDFs by batching ingestion
- Use `ConversationalRetrievalChain` for multi-turn memory
